In [ ]:
#This file does the statistical analysis of the BERT predictions and metadata as well as visualizations
#importing libraries
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Logit
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, ScalarFormatter

In [ ]:
#Claude did error/exception handling, debugging
def analyze_antisemitism_patterns(data_file, confidence_threshold=None):
    #subreddit categories
    right_wing_subreddits = [
        "conspiracy", "conservative", "askaconservative", "AskConservatives", "conservatives",
        "4chan", "politicalcompassmemes", "conservativememes", "conspiracy_commons", "darkenlightenment"
    ]

    mainstream_subreddits = [
        "worldnews", "politics", "news", "InternationalNews", "AnythingGoesNews"
    ]

    left_wing_subreddits = [
        "LateStageCapitalism", "antiwork", "Palestine", "socialism", "Socialism101",
        "Hasan_Piker", "gaza"
    ]

    #loading data
    df = pd.read_csv(data_file)

    #confidence threshold filtering
    if confidence_threshold is not None:
        original_count = len(df)
        df = df[df['prediction_confidence'] >= confidence_threshold]
        filtered_count = len(df)
        print(f"\nFiltered out {original_count - filtered_count} entries with confidence < {confidence_threshold}")
        print(f"Remaining entries: {filtered_count} ({filtered_count/original_count*100:.1f}% of original data)")

    #map subreddits to orientations
    def get_orientation(subreddit):
        if not isinstance(subreddit, str):
            return 'unknown'

        subreddit_lower = subreddit.lower()

        #check if in right wing list
        if subreddit_lower in [s.lower() for s in right_wing_subreddits]:
            return 'right'

        #check if in mainstream list
        if subreddit_lower in [s.lower() for s in mainstream_subreddits]:
            return 'neutral'

        #check if in left wing list
        if subreddit_lower in [s.lower() for s in left_wing_subreddits]:
            return 'left'

        return 'unknown'

    #add orientation column
    df['subreddit_orientation'] = df['subreddit'].apply(get_orientation)

    #filter out unknown orientations
    unknown_count = df[df['subreddit_orientation'] == 'unknown'].shape[0]
    if unknown_count > 0:
        print(f"Found {unknown_count} entries with unknown orientation. These will be excluded from analysis.")
        df = df[df['subreddit_orientation'] != 'unknown']

    #rename predicted_category to category for clarity
    df['category'] = df['predicted_category'].astype(int)

    #filter out category 4 ("None of the above") as it's not necessary for analysis
    category_4_count = df[df['category'] == 4].shape[0]
    if category_4_count > 0:
        print(f"Removing {category_4_count} entries with category 4 (None of the above)")
        df = df[df['category'] != 4]

    #datetime conversion
    df['post_date'] = pd.to_datetime(df['created_utc'])
    df['post_year'] = df['post_date'].dt.year
    df['post_month'] = df['post_date'].dt.month

    #ensure score is numeric
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    # Fill any NaN values in score with the median score
    df['score'].fillna(df['score'].median(), inplace=True)

    ######1. Basic descriptive statistics
    print("\n--- Descriptive Statistics ---")
    print("\nOverall category distribution:")
    category_counts = df['category'].value_counts().sort_index()
    category_percent = df['category'].value_counts(normalize=True).sort_index() * 100

    for cat, count in category_counts.items():
        cat_name = ""
        if cat == 1:
            cat_name = "Traditional antisemitism"
        elif cat == 2:
            cat_name = "Israel critique, not antisemitic"
        elif cat == 3:
            cat_name = "Israel-based antisemitism"

        print(f"Category {cat} ({cat_name}): {count} posts ({category_percent[cat]:.1f}%)")

    #distribution by subreddit orientation
    print("\nSubreddit orientation distribution:")
    orientation_counts = df['subreddit_orientation'].value_counts()
    for orient, count in orientation_counts.items():
        print(f"{orient}: {count} posts ({count/len(df)*100:.1f}%)")

    ######2. Crosstabulation of antisemitism category by subreddit orientation
    print("\n--- Cross-tabulation of Category by Orientation ---")

    #crosstab
    ctab = pd.crosstab(df['subreddit_orientation'], df['category'])
    ctab['Total'] = ctab.sum(axis=1)
    ctab.loc['Total'] = ctab.sum()
    ctab_pct = pd.crosstab(df['subreddit_orientation'], df['category'], normalize='index') * 100

    print("\nCounts:")
    print(ctab)

    print("\nPercentages (row %):")
    print(ctab_pct.round(1))

    ######3. Chi-square test for independence
    print("\n--- Chi-Square Test for Independence ---")

    #create the contingency tab
    contingency = pd.crosstab(df['subreddit_orientation'], df['category']).values

    #chi-square test
    chi2, p, dof, expected = stats.chi2_contingency(contingency)

    print(f"Chi-square statistic: {chi2:.2f}")
    print(f"p-value: {p:.4f}")
    print(f"Degrees of freedom: {dof}")

    if p < 0.05:
        print("Result: There is a significant association between subreddit orientation and antisemitism category")
    else:
        print("Result: No significant association between subreddit orientation and antisemitism category")

    #####4. Logistic Regression to examine specific relationships
    print("\n--- Logistic Regression Analysis ---")

    #create binary outcome variables for specific comparisons
    #Traditional antisemitism (Category 1)
    df['is_traditional'] = (df['category'] == 1).astype(int)
    #Israel critique, not antisemitic (Category 2)
    df['is_israel_critique'] = (df['category'] == 2).astype(int)
    #Israel-based antisemitism (Category 3)
    df['is_israel_antisemitism'] = (df['category'] == 3).astype(int)

    #dummy variables for subreddit orientation (with 'neutral' as baseline)
    df = pd.get_dummies(df, columns=['subreddit_orientation'], drop_first=False)

    model_results = {}

    #Claude added the odds ratio-relatedd code
    #Traditional antisemitism model (Model 1)
    print("\nModel 1: Predicting Traditional Antisemitism (Category 1)")

    #without controls
    print("\nModel 1a: Without controls")
    X1a = df[['subreddit_orientation_left', 'subreddit_orientation_right']].astype(float)
    X1a = sm.add_constant(X1a)
    y1 = df['is_traditional'].astype(float)

    try:
        model1a = sm.Logit(y1, X1a).fit(disp=0)
        print(model1a.summary2())

        #pseudo R-squared
        null_model1 = sm.Logit(y1, np.ones(len(y1))).fit(disp=0)
        pseudo_r2_1a = 1 - model1a.llf / null_model1.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_1a:.4f}")

        #convert log odds to odds ratios
        odds_ratios1a = pd.DataFrame({
            'Odds Ratio': np.exp(model1a.params),
            '95% CI Lower': np.exp(model1a.conf_int()[0]),
            '95% CI Upper': np.exp(model1a.conf_int()[1]),
            'p-value': model1a.pvalues
        })
        print("\nOdds Ratios for Traditional Antisemitism (Without controls):")
        print(odds_ratios1a)

        model_results['model1a'] = {'model': model1a, 'odds_ratios': odds_ratios1a, 'pseudo_r2': pseudo_r2_1a}
    except Exception as e:
        print(f"Error in Model 1a: {e}")

    #with controls
    print("\nModel 1b: With controls")
    X1b = df[['subreddit_orientation_left', 'subreddit_orientation_right',
             'post_year', 'post_month', 'score']].astype(float)
    X1b = sm.add_constant(X1b)

    try:
        model1b = sm.Logit(y1, X1b).fit(disp=0)
        print(model1b.summary2())

        #pseudo R-squared
        pseudo_r2_1b = 1 - model1b.llf / null_model1.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_1b:.4f}")

        #convert log odds to odds ratios
        odds_ratios1b = pd.DataFrame({
            'Odds Ratio': np.exp(model1b.params),
            '95% CI Lower': np.exp(model1b.conf_int()[0]),
            '95% CI Upper': np.exp(model1b.conf_int()[1]),
            'p-value': model1b.pvalues
        })
        print("\nOdds Ratios for Traditional Antisemitism (With controls):")
        print(odds_ratios1b)

        model_results['model1b'] = {'model': model1b, 'odds_ratios': odds_ratios1b, 'pseudo_r2': pseudo_r2_1b}
    except Exception as e:
        print(f"Error in Model 1b: {e}")

    #Israel critique, not antisemitic model (Model 2)
    print("\nModel 2: Predicting Israel Critique, Not Antisemitic (Category 2)")

    #without controls
    print("\nModel 2a: Without controls")
    X2a = df[['subreddit_orientation_left', 'subreddit_orientation_right']].astype(float)
    X2a = sm.add_constant(X2a)
    y2 = df['is_israel_critique'].astype(float)

    try:
        model2a = sm.Logit(y2, X2a).fit(disp=0)
        print(model2a.summary2())

        #pseudo R-squared
        null_model2 = sm.Logit(y2, np.ones(len(y2))).fit(disp=0)
        pseudo_r2_2a = 1 - model2a.llf / null_model2.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_2a:.4f}")

        #convert log odds to odds ratios
        odds_ratios2a = pd.DataFrame({
            'Odds Ratio': np.exp(model2a.params),
            '95% CI Lower': np.exp(model2a.conf_int()[0]),
            '95% CI Upper': np.exp(model2a.conf_int()[1]),
            'p-value': model2a.pvalues
        })
        print("\nOdds Ratios for Israel Critique, Not Antisemitic (Without controls):")
        print(odds_ratios2a)

        model_results['model2a'] = {'model': model2a, 'odds_ratios': odds_ratios2a, 'pseudo_r2': pseudo_r2_2a}
    except Exception as e:
        print(f"Error in Model 2a: {e}")

    #with controls
    print("\nModel 2b: With controls")
    X2b = df[['subreddit_orientation_left', 'subreddit_orientation_right',
             'post_year', 'post_month', 'score']].astype(float)
    X2b = sm.add_constant(X2b)

    try:
        model2b = sm.Logit(y2, X2b).fit(disp=0)
        print(model2b.summary2())

        #pseudo R-squared
        pseudo_r2_2b = 1 - model2b.llf / null_model2.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_2b:.4f}")

        #convert log odds to odds ratios
        odds_ratios2b = pd.DataFrame({
            'Odds Ratio': np.exp(model2b.params),
            '95% CI Lower': np.exp(model2b.conf_int()[0]),
            '95% CI Upper': np.exp(model2b.conf_int()[1]),
            'p-value': model2b.pvalues
        })
        print("\nOdds Ratios for Israel Critique, Not Antisemitic (With controls):")
        print(odds_ratios2b)

        model_results['model2b'] = {'model': model2b, 'odds_ratios': odds_ratios2b, 'pseudo_r2': pseudo_r2_2b}
    except Exception as e:
        print(f"Error in Model 2b: {e}")

    #Israel-based antisemitism model (Model 3)
    print("\nModel 3: Predicting Israel-based Antisemitism (Category 3)")

    #without controls
    print("\nModel 3a: Without controls")
    X3a = df[['subreddit_orientation_left', 'subreddit_orientation_right']].astype(float)
    X3a = sm.add_constant(X3a)
    y3 = df['is_israel_antisemitism'].astype(float)

    try:
        model3a = sm.Logit(y3, X3a).fit(disp=0)
        print(model3a.summary2())

        #pseudo R-squared
        null_model3 = sm.Logit(y3, np.ones(len(y3))).fit(disp=0)
        pseudo_r2_3a = 1 - model3a.llf / null_model3.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_3a:.4f}")

        #convert log odds to odds ratios
        odds_ratios3a = pd.DataFrame({
            'Odds Ratio': np.exp(model3a.params),
            '95% CI Lower': np.exp(model3a.conf_int()[0]),
            '95% CI Upper': np.exp(model3a.conf_int()[1]),
            'p-value': model3a.pvalues
        })
        print("\nOdds Ratios for Israel-based Antisemitism (Without controls):")
        print(odds_ratios3a)

        model_results['model3a'] = {'model': model3a, 'odds_ratios': odds_ratios3a, 'pseudo_r2': pseudo_r2_3a}
    except Exception as e:
        print(f"Error in Model 3a: {e}")

    #with controls
    print("\nModel 3b: With controls")
    X3b = df[['subreddit_orientation_left', 'subreddit_orientation_right',
             'post_year', 'post_month', 'score']].astype(float)
    X3b = sm.add_constant(X3b)

    try:
        model3b = sm.Logit(y3, X3b).fit(disp=0)
        print(model3b.summary2())

        #pseudo R-squared
        pseudo_r2_3b = 1 - model3b.llf / null_model3.llf
        print(f"McFadden's Pseudo R-squared: {pseudo_r2_3b:.4f}")

        #convert log odds to odds ratios
        odds_ratios3b = pd.DataFrame({
            'Odds Ratio': np.exp(model3b.params),
            '95% CI Lower': np.exp(model3b.conf_int()[0]),
            '95% CI Upper': np.exp(model3b.conf_int()[1]),
            'p-value': model3b.pvalues
        })
        print("\nOdds Ratios for Israel-based Antisemitism (With controls):")
        print(odds_ratios3b)

        model_results['model3b'] = {'model': model3b, 'odds_ratios': odds_ratios3b, 'pseudo_r2': pseudo_r2_3b}
    except Exception as e:
        print(f"Error in Model 3b: {e}")

    print("\nAnalysis complete.")
    return df, model_results

In [ ]:
analyze_antisemitism_patterns("/content/drive/MyDrive/bert_predictions.csv", confidence_threshold=0.5)

Loading data...

Dataset shape: (9013, 16)

Columns in the dataset:
['category_code', 'subreddit', 'search_term', 'post_id', 'comment_id', 'parent_id', 'body', 'created_utc', 'score', 'permalink', 'url', 'author', 'is_comment', 'predicted_category', 'prediction_confidence', 'predicted_category_name']

Filtered out 2086 entries with confidence < 0.5
Remaining entries: 6927 (76.9% of original data)

Mapping subreddits to political orientations...
Removing 1423 entries with category 4 (None of the above)

--- Descriptive Statistics ---

Overall category distribution:
Category 1 (Traditional antisemitism): 1455 posts (26.4%)
Category 2 (Israel critique, not antisemitic): 2525 posts (45.9%)
Category 3 (Israel-based antisemitism): 1524 posts (27.7%)

Subreddit orientation distribution:
neutral: 3118 posts (56.6%)
right: 1357 posts (24.7%)
left: 1029 posts (18.7%)

--- Cross-tabulation of Category by Orientation ---

Counts:
category                  1     2     3  Total
subreddit_orientation

<ipython-input-7-4d84254b6d1d>:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['score'].fillna(df['score'].median(), inplace=True)


                               Results: Logit
Model:                   Logit               Method:              MLE       
Dependent Variable:      is_traditional      Pseudo R-squared:    0.063     
Date:                    2025-04-24 20:31    AIC:                 5961.3801 
No. Observations:        5504                BIC:                 5981.2198 
Df Model:                2                   Log-Likelihood:      -2977.7   
Df Residuals:            5501                LL-Null:             -3178.9   
Converged:               1.0000              LLR p-value:         4.1411e-88
No. Iterations:          7.0000              Scale:               1.0000    
----------------------------------------------------------------------------
                             Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
----------------------------------------------------------------------------
const                       -1.0371   0.0407 -25.4559 0.0000 -1.1170 -0.9573
subreddit_orientation_left  -1

(      category_code            subreddit      search_term     post_id  \
 1                 0  LateStageCapitalism  apartheid state  t3_191uh1g   
 2                 0  LateStageCapitalism  apartheid state  t3_190yd97   
 4                 0  LateStageCapitalism  apartheid state  t3_18j0cb0   
 5                 0  LateStageCapitalism  apartheid state  t3_18j0cb0   
 6                 0  LateStageCapitalism  apartheid state  t3_18j0cb0   
 ...             ...                  ...              ...         ...   
 9006              2    darkenlightenment             1488   t3_4ef83h   
 9008              2    darkenlightenment  jewish question   t3_3d53a6   
 9009              2    darkenlightenment  jewish question   t3_3d53a6   
 9010              2    darkenlightenment  jewish question   t3_3d53a6   
 9012              2    darkenlightenment  jewish question   t3_1z773t   
 
      comment_id   parent_id  \
 1       khdlzxw  t3_191uh1g   
 2       kgsbudb  t1_kgrf9go   
 4       kdk05

<Figure size 1400x800 with 0 Axes>

In [ ]:
#Claude and Copilot did a lot of debugging here and Claude suggested the odds ratio visualization
def create_improved_visualizations(df, model_results):

    #output directory
    os.makedirs("visualizations", exist_ok=True)

#####1. bar chart for subreddits, colored by antisemitism types
    plt.figure(figsize=(14, 8))

    #obtain percentage of each AS type within each subreddit ideology group
    orientation_map = {}
    for idx, row in df.iterrows():
        if row['subreddit_orientation_left']:
            orientation_map[idx] = 'Left'
        elif row['subreddit_orientation_right']:
            orientation_map[idx] = 'Right'
        else:
            orientation_map[idx] = 'Neutral'

    df['orientation'] = pd.Series(orientation_map)

    #create crosstab
    plot_df = pd.crosstab(df['orientation'], df['category'], normalize='index') * 100

    #reorder columns to match the category numbers
    if not all(i in plot_df.columns for i in [1, 2, 3]):
        missing_cols = [i for i in [1, 2, 3] if i not in plot_df.columns]
        for col in missing_cols:
            plot_df[col] = 0
    plot_df = plot_df[[1, 2, 3]]

    #set pretty colors
    colors = ['#66c2a5', '#fc8d62', '#8da0cb']  # green, orange, muted blue

    #set up the plot
    ax = plot_df.plot(kind='bar', width=0.8, color=colors, figsize=(14, 8))
    plt.title('Types of Antisemitism by Subreddit Political Orientation', fontsize=16)
    plt.xlabel('Subreddit Orientation', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.legend(['Traditional Antisemitism',
                'Israel Critique (Not Antisemitic)',
                'Israel Critique with Antisemitism'],
              fontsize=12, loc='best')
    plt.xticks(rotation=0, fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    #add percentage labels on bars
    for i, p in enumerate(ax.patches):
        width, height = p.get_width(), p.get_height()
        x, y = p.get_xy()
        ax.annotate(f'{height:.1f}%',
                   (x + width/2, y + height + 1),
                   ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('visualizations/antisemitism_by_orientation_new_colors.png', dpi=300)
    plt.close()

    ######2.coefficient heatmap with significance indicators
    try:
        #extract coefficients and p-values from all models
        model_coefs = {}
        model_pvals = {}

        for model_name, model_label in [
            ('model1b', 'Traditional\nAntisemitism'),
            ('model2b', 'Israel Critique\n(Not Antisemitic)'),
            ('model3b', 'Israel-based\nAntisemitism')
        ]:
            if model_name in model_results:
                model = model_results[model_name]['model']

                #get coefficients excluding constant
                coefs = {}
                pvals = {}
                for var in model.params.index:
                    if var != 'const':
                        var_name = var.replace('subreddit_orientation_', '').capitalize()
                        if var == 'post_year':
                            var_name = 'Year'
                        elif var == 'post_month':
                            var_name = 'Month'
                        elif var == 'score':
                            var_name = 'Score'

                        coefs[var_name] = model.params[var]
                        pvals[var_name] = model.pvalues[var]

                model_coefs[model_label] = coefs
                model_pvals[model_label] = pvals

        #convert to DataFrame
        coef_df = pd.DataFrame(model_coefs)
        pval_df = pd.DataFrame(model_pvals)

        #create a diverging colormap centered at 0 #courtesy of ChatGPT
        colors = ["#d73027", "#f46d43", "#fdae61", "#fee090", "#ffffff",
                  "#e0f3f8", "#abd9e9", "#74add1", "#4575b4"]
        cmap = LinearSegmentedColormap.from_list("custom_diverging", colors, N=256)

        plt.figure(figsize=(10, 8))
        ax = sns.heatmap(
            coef_df,
            cmap=cmap,
            center=0,
            annot=True,
            fmt=".2f",
            linewidths=.5,
            cbar_kws={"label": "Coefficient (Log Odds)"}
        )

        #add significance stars
        for i, var in enumerate(coef_df.index):
            for j, model in enumerate(coef_df.columns):
                if var in pval_df.index and model in pval_df.columns:
                    p_value = pval_df.loc[var, model]
                    stars = ''
                    if p_value < 0.05:
                        stars = '*'
                    if p_value < 0.01:
                        stars = '**'
                    if p_value < 0.001:
                        stars = '***'
                    if stars:
                        ax.text(j + 0.5, i + 0.85, stars, ha='center', va='top', color='black', fontweight='bold')

        plt.title('Coefficient Heatmap Across Models', fontsize=16)
        plt.tight_layout()
        plt.savefig('visualizations/coefficient_heatmap.png', dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error creating heatmap: {e}")

    #####3. R² comparison
    try:
        #collect values
        r2_data = []
        for model_key, model_name in [
            ('model1a', 'Traditional (Basic)'),
            ('model1b', 'Traditional (Controls)'),
            ('model2a', 'Israel Critique (Basic)'),
            ('model2b', 'Israel Critique (Controls)'),
            ('model3a', 'Israel-based (Basic)'),
            ('model3b', 'Israel-based (Controls)')
        ]:
            if model_key in model_results:
                r2_data.append({
                    'Model': model_name,
                    'R²': model_results[model_key]['pseudo_r2'],
                    'Type': 'Basic' if 'Basic' in model_name else 'Controls'
                })

        r2_df = pd.DataFrame(r2_data)

        #sort by model name
        model_order = [
            'Traditional (Basic)', 'Traditional (Controls)',
            'Israel Critique (Basic)', 'Israel Critique (Controls)',
            'Israel-based (Basic)', 'Israel-based (Controls)'
        ]
        r2_df['Model'] = pd.Categorical(r2_df['Model'], categories=model_order, ordered=True)
        r2_df = r2_df.sort_values('Model')

        plt.figure(figsize=(12, 6))
        bar_colors = ['#66c2a5', '#66c2a5', '#fc8d62', '#fc8d62', '#8da0cb', '#8da0cb']
        alpha = 0.5  #transparency thing

        bars = plt.bar(r2_df['Model'], r2_df['R²'], color=bar_colors, alpha=alpha)

        #add value labels
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=10)

        #add threshold line
        plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.5)

        plt.title('Pseudo R² Comparison Across Models', fontsize=16)
        plt.ylabel('McFadden\'s Pseudo R²', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        plt.ylim(0, max(r2_df['R²']) * 1.2)

        plt.tight_layout()
        plt.savefig('visualizations/pseudo_r2_comparison.png', dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error creating R² comparison: {e}")

    #######4.odds ratios comparison (by Claude)
    try:
        fig, axes = plt.subplots(3, 1, figsize=(10, 15))

        models = [
            ('model1b', 'Traditional Antisemitism', axes[0], '#66c2a5'),
            ('model2b', 'Israel Critique (Not Antisemitic)', axes[1], '#fc8d62'),
            ('model3b', 'Israel-based Antisemitism', axes[2], '#8da0cb')
        ]

        #plot each model's odds ratios
        for model_key, title, ax, color in models:
            if model_key in model_results:
                odds_df = model_results[model_key]['odds_ratios']
                variables = ['subreddit_orientation_left', 'subreddit_orientation_right']

                odds = odds_df.loc[variables, 'Odds Ratio']
                lower = odds_df.loc[variables, '95% CI Lower']
                upper = odds_df.loc[variables, '95% CI Upper']
                pvals = odds_df.loc[variables, 'p-value']

                #error bars
                yerr = np.zeros((2, 2))
                yerr[0, :] = odds - lower
                yerr[1, :] = upper - odds

                x = [0, 1]
                ax.errorbar(x, odds, yerr=yerr, fmt='o', capsize=5, color=color, markersize=10)

                #add line at odds ratio = 1
                ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7)

                #add value labels and significance
                for i, (val, p) in enumerate(zip(odds, pvals)):
                    #significance markers
                    if p < 0.001:
                        sig = '***'
                    elif p < 0.01:
                        sig = '**'
                    elif p < 0.05:
                        sig = '*'
                    else:
                        sig = 'ns'

                    offset = 0.10 * val if val > 1 else -0.10 * val #text was overlapping confidence intervals at first
                    va_align = 'bottom' if val > 1 else 'top'
                    ax.text(x[i], val + offset, f"{val:.2f}\n{sig}",
                        ha='center', va=va_align, fontsize=12, fontweight='bold')

                #set axis properties
                ax.set_xticks(x)
                ax.set_xticklabels(['Left', 'Right'], fontsize=12)
                ax.set_yscale('log') # Set y-axis to logarithmic scale
                ax.grid(True, alpha=0.3)

                #add pseudo R²
                r2 = model_results[model_key]['pseudo_r2']
                ax.set_title(f"{title}\nPseudo R² = {r2:.3f}", fontsize=14)

                #set y-axis limits to better show data
                y_min = min(lower) * 0.8
                y_max = max(upper) * 1.2
                ax.set_ylim(y_min, y_max)

                #add y-axis label
                ax.set_ylabel('Odds Ratio (vs. Neutral)', fontsize=12)

        #add overall title and legend
        plt.figtext(0.5, 0.02, '* p < 0.05, ** p < 0.01, *** p < 0.001, ns: not significant',
                  ha='center', fontsize=12)

        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig('visualizations/odds_ratios_improved.png', dpi=300)
        plt.close()
    except Exception as e:
        print(f"Error creating odds ratio plot: {e}")

    print("\nImproved visualizations created successfully.")
    return True

In [ ]:
df, model_results = analyze_antisemitism_patterns("/content/drive/MyDrive/bert_predictions.csv", confidence_threshold=0.5)
create_improved_visualizations(df, model_results)

Loading data...

Dataset shape: (9013, 16)

Columns in the dataset:
['category_code', 'subreddit', 'search_term', 'post_id', 'comment_id', 'parent_id', 'body', 'created_utc', 'score', 'permalink', 'url', 'author', 'is_comment', 'predicted_category', 'prediction_confidence', 'predicted_category_name']

Filtered out 2086 entries with confidence < 0.5
Remaining entries: 6927 (76.9% of original data)

Mapping subreddits to political orientations...
Removing 1423 entries with category 4 (None of the above)

--- Descriptive Statistics ---

Overall category distribution:
Category 1 (Traditional antisemitism): 1455 posts (26.4%)
Category 2 (Israel critique, not antisemitic): 2525 posts (45.9%)
Category 3 (Israel-based antisemitism): 1524 posts (27.7%)

Subreddit orientation distribution:
neutral: 3118 posts (56.6%)
right: 1357 posts (24.7%)
left: 1029 posts (18.7%)

--- Cross-tabulation of Category by Orientation ---

Counts:
category                  1     2     3  Total
subreddit_orientation

<ipython-input-2-51ec19994077>:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['score'].fillna(df['score'].median(), inplace=True)


                                Results: Logit
Model:                   Logit                Method:               MLE       
Dependent Variable:      is_traditional       Pseudo R-squared:     0.073     
Date:                    2025-04-24 21:56     AIC:                  5908.8337 
No. Observations:        5504                 BIC:                  5948.5130 
Df Model:                5                    Log-Likelihood:       -2948.4   
Df Residuals:            5498                 LL-Null:              -3178.9   
Converged:               1.0000               LLR p-value:          2.1235e-97
No. Iterations:          7.0000               Scale:                1.0000    
------------------------------------------------------------------------------
                             Coef.   Std.Err.    z    P>|z|   [0.025   0.975] 
------------------------------------------------------------------------------
const                       137.7462  18.4403  7.4698 0.0000 101.6039 173.8885
subre

True

<Figure size 1400x800 with 0 Axes>